### Load files

In [2]:
import pandas as pd

A = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\20-21\ASI_DATA_2020_21_CSV\blkA202021.csv")
B = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\20-21\ASI_DATA_2020_21_CSV\blkB202021.csv")
E = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\20-21\ASI_DATA_2020_21_CSV\blkE202021.csv")
F = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\20-21\ASI_DATA_2020_21_CSV\blkF202021.csv")
G = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\20-21\ASI_DATA_2020_21_CSV\blkG202021.csv")
J = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\20-21\ASI_DATA_2020_21_CSV\blkJ202021.csv")

### Check column names

In [4]:
for name, df in {
    "A":A,
    "B":B,
    "E":E,
    "F":F,
    "G":G,
    "J":J
}.items():
    print("\n",name)
    print(df.columns.tolist())


 A
['yr', 'blk', 'a1', 'a2', 'a3', 'a4', 'a5', 'a7', 'a8', 'a9', 'a10', 'a11', 'a12', 'bonus', 'pf', 'welfare', 'mwdays', 'nwdays', 'wdays', 'costop', 'expshare', 'mult']

 B
['yr', 'blk', 'ab01', 'b02', 'b03', 'b04', 'b05', 'b06f', 'b06t', 'b07', 'b08', 'b09', 'b11']

 E
['yr', 'blk', 'AE01', 'E11', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18']

 F
['yr', 'blk', 'AF01', 'F1', 'F2A', 'F2B', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12', 'F13']

 G
['yr', 'blk', 'AG01', 'G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9', 'G10', 'G11', 'G12']

 J
['yr', 'blk', 'AJ01', 'J11', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J110', 'J111', 'J112', 'J113']


### CBAM filter

In [5]:
cbam_codes = [
    24101,24102,24103,24104,24105,24106,24107,
    24201,24202,24203,24204,24209,
    23941,23942,
    20121,20122,20129,20130
]

cbam = A[A["a5"].isin(cbam_codes)]

print(cbam.shape)
print(cbam["a5"].value_counts())

(2335, 22)
a5
24105    382
24103    259
24202    249
24209    188
23942    184
24102    148
20129    126
24101    121
23941    116
24201    104
24203     94
24104     91
24106     90
20121     86
20122     62
24204     22
24107     13
Name: count, dtype: int64


### Exporters only

In [6]:
cbam_exporters = cbam[
    cbam["expshare"] > 0
]

print(cbam_exporters.shape)

print(
    cbam_exporters["expshare"].describe()
)

print(
    cbam_exporters[
        ["a1","a5","expshare"]
    ].sort_values(
        "expshare",
        ascending=False
    ).head(20)
)

(130, 22)
count    130.000000
mean      30.846154
std       30.728988
min        1.000000
25%        7.250000
50%       19.500000
75%       46.000000
max      100.000000
Name: expshare, dtype: float64
           a1     a5  expshare
32172  137278  24209       100
20015  122242  24105        98
29074  133312  24209        93
19351  121471  24104        92
34085  140000  24209        91
34079  139994  24201        91
55514  211025  24204        90
20041  122273  24105        88
20057  122290  24105        88
20099  122344  24105        88
20007  122232  24105        88
20064  122299  24105        88
20069  122305  24105        88
20094  122339  24102        88
20052  122285  24102        88
20011  122238  24105        88
20020  122247  24105        88
20040  122272  24103        88
20061  122296  24102        88
20076  122314  24105        88


### Employment aggregation

In [7]:
emp = (
    E.groupby("AE01")
     .sum(numeric_only=True)
     .reset_index()
)

print(emp.head())

     AE01   yr  E11    E13  E14    E15  E16    E17        E18
0  100001  189   45      0  720    720    2    730   300000.0
1  100002  189   45  25962    0  25962   91  27032  8791350.0
2  100003  189   45   9248    0   9248   53   9400  3297870.0
3  100004  189   45   5108    0   5108   18   7148  2989640.0
4  100005  189   45  16483    0  16483   61  18189  7060927.0


### Output aggregation

In [8]:
output = (
    J.groupby("AJ01")["J113"]
     .sum()
     .reset_index()
)

output.columns = [
    "factory",
    "output_value"
]

print(output.head())

   factory  output_value
0   100002    62148600.0
1   100003   202633808.0
2   100004    36118698.0
3   100005    64275456.0
4   100006     6741556.0


### Merge

In [9]:
master = cbam_exporters.merge(
    emp,
    left_on="a1",
    right_on="AE01",
    how="left"
)

master = master.merge(
    output,
    left_on="a1",
    right_on="factory",
    how="left"
)

print(master.shape)
print(master.head())

(130, 33)
   yr_x blk      a1     a2  a3    a4     a5  a7  a8  a9  ...  yr_y  E11  \
0    21   A  102006  99999   1  9999  24103   3  99   2  ...   189   45   
1    21   A  102008  99999   1  9999  24105   3  99   2  ...   189   45   
2    21   A  102215  99999   1  9999  24201   3  99   2  ...   189   45   
3    21   A  105350  99999   1  9999  24105   6  99   2  ...   189   45   
4    21   A  105371  99999   1  9999  24103   6  99   2  ...   189   45   

       E13  E14      E15    E16      E17           E18  factory  output_value  
0   376843    0   376843   1352   431055  3.210155e+08   102006  2.189022e+09  
1   213532    0   213532    760   296122  1.642907e+08   102008  4.049132e+09  
2   218202    0   218202    724   292130  1.196806e+08   102215  9.004033e+08  
3  5882754    0  5882754  21006  6099676  4.396562e+09   105350  1.297114e+11  
4   175272    0   175272    654   238710  2.162000e+08   105371  1.919763e+09  

[5 rows x 33 columns]


### Missing-value check

In [10]:
print(
    master[
        ["E13","E15","E18","output_value"]
    ].isna().sum()
)

E13             0
E15             0
E18             0
output_value    0
dtype: int64


### Save

In [11]:
master.to_excel(
    "CBAM_Exporters_2020_21.xlsx",
    index=False
)

### Verify whether you have Block I for ALL years

In [12]:
import pandas as pd

I = pd.read_csv(r"C:\Users\hp\Downloads\CBAM PROJECT\20-21\ASI_DATA_2020_21_CSV\blkI202021.csv")
print(I.columns.tolist())
print(I.head())

['yr', 'blk', 'AI01', 'I11', 'I13', 'I14', 'I15', 'I16', 'I17']
   yr blk    AI01  I11      I13  I14          I15           I16        I17
0  21   I  100066    1   164000    9  22588816.33  4.360997e+09     193.06
1  21   I  100066    7  9994000    0         0.00  4.360997e+09       0.00
2  21   I  100075    1   137600   27      1009.00  3.456235e+08  342540.61
3  21   I  100075    7  9994000    0         0.00  3.456235e+08       0.00
4  21   I  100079    1   137600   27       593.00  2.052568e+08  346132.84


In [13]:
print(sorted(I["I13"].unique())[:100])

[np.int64(112100), np.int64(112200), np.int64(115200), np.int64(117100), np.int64(117200), np.int64(118200), np.int64(121300), np.int64(121900), np.int64(123100), np.int64(123200), np.int64(123500), np.int64(124900), np.int64(125100), np.int64(125300), np.int64(125400), np.int64(126000), np.int64(129003), np.int64(129099), np.int64(131400), np.int64(131900), np.int64(132300), np.int64(133000), np.int64(134900), np.int64(135200), np.int64(135300), np.int64(135400), np.int64(135900), np.int64(136000), np.int64(137100), np.int64(137200), np.int64(137400), np.int64(137500), np.int64(137600), np.int64(137900), np.int64(139100), np.int64(141200), np.int64(144400), np.int64(144500), np.int64(144901), np.int64(144999), np.int64(145000), np.int64(146000), np.int64(149100), np.int64(161001), np.int64(162002), np.int64(164000), np.int64(165100), np.int64(165200), np.int64(165300), np.int64(165400), np.int64(165500), np.int64(165600), np.int64(165700), np.int64(165800), np.int64(165900), np.int64(

In [14]:
fuel20 = (
    I.groupby("I13")["I16"]
    .sum()
    .sort_values(ascending=False)
)

fuel20.head(30)

I13
9994000    1.236300e+13
1201000    4.119440e+12
1639001    8.562072e+11
1101002    6.958516e+11
9922100    3.928542e+11
2153500    3.678873e+11
1632001    2.242862e+11
1202003    2.189136e+11
2153100    1.763660e+11
1421000    1.510539e+11
3337000    1.323263e+11
3411071    1.207785e+11
4912999    1.123646e+11
2153300    1.053687e+11
4740100    1.023899e+11
1424004    1.021068e+11
3423200    7.191827e+10
3936102    6.707929e+10
4714099    6.646532e+10
4141202    6.098971e+10
4132003    5.754240e+10
3934003    5.537930e+10
4121101    5.315425e+10
1720099    5.209564e+10
1101006    4.739030e+10
3549099    4.519310e+10
1639006    4.426999e+10
3824099    4.271921e+10
3935001    3.802888e+10
3418001    3.792282e+10
Name: I16, dtype: float64

In [16]:
codes = pd.DataFrame(
    sorted(I["I13"].dropna().unique()),
    columns=["item_code"]
)

codes.to_excel("ASI_I_block_codes20.xlsx", index=False)

print("Saved")

Saved


In [18]:
for name, df in {
    "A":A,
    "E":E,
    "F":F,
    "G":G,
    "I":I,
    "J":J
}.items():

    print("\n", "="*50)
    print(name)

    for col in df.columns:
        if "exp" in str(col).lower():
            print(col)
for name, df in {
    "A":A,
    "E":E,
    "F":F,
    "G":G,
    "I":I,
    "J":J
}.items():

    print("\n", name)

    for col in df.columns:
        print(col)


A
expshare

E

F

G

I

J

 A
yr
blk
a1
a2
a3
a4
a5
a7
a8
a9
a10
a11
a12
bonus
pf
welfare
mwdays
nwdays
wdays
costop
expshare
mult

 E
yr
blk
AE01
E11
E13
E14
E15
E16
E17
E18

 F
yr
blk
AF01
F1
F2A
F2B
F3
F4
F5
F6
F7
F8
F9
F10
F11
F12
F13

 G
yr
blk
AG01
G1
G2
G3
G4
G5
G6
G7
G8
G9
G10
G11
G12

 I
yr
blk
AI01
I11
I13
I14
I15
I16
I17

 J
yr
blk
AJ01
J11
J13
J14
J15
J16
J17
J18
J19
J110
J111
J112
J113
